# DCSASS dataset inspection
In Colab select **Runtime → Change runtime type → T4 GPU** (optional for inspection), then **Runtime → Run all**. Mount Drive when prompted. Run cells in order. Publish the repository changes first so cloning includes these utilities. No dataset statistics are assumed. Each unique canonical video is decoded once; duplicate copies are streamed for hashing; a readable first frame does not prove the entire clip is valid.
Schema v2: regenerate old manifests. Canonical root deduplication verifies duplicate bytes with SHA-256 and writes excluded_duplicates.csv. Only Burglary, Stealing, Fighting, and Vandalism are mapped. Normal and all other categories remain separately reported as unmapped. No Colab or real-data result is claimed by this notebook.


In [ ]:
from pathlib import Path
import os, subprocess, sys
REPO = Path('/content/AI-Crime-Investigation-FYP')
if not REPO.exists():
    subprocess.run(['git', 'clone', 'https://github.com/ranaalihaider99993-source/AI-Crime-Investigation-FYP.git', str(REPO)], check=True)
else:
    if subprocess.check_output(['git', '-C', str(REPO), 'status', '--porcelain'], text=True).strip():
        raise RuntimeError('Colab checkout has local changes. Preserve them before updating; use a fresh runtime if needed.')
    subprocess.run(['git', '-C', str(REPO), 'fetch', 'origin', 'main'], check=True)
    subprocess.run(['git', '-C', str(REPO), 'merge', '--ff-only', 'origin/main'], check=True)
os.chdir(REPO)
# Evict previously imported project modules after updating a reused Colab runtime.
for module_name in list(sys.modules):
    if module_name == 'scripts' or module_name.startswith('scripts.'):
        del sys.modules[module_name]
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', 'requirements.txt'], check=True)
import torch
from scripts.common import seed_all
seed_all(42)
print('Device:', 'cuda' if torch.cuda.is_available() else 'cpu')
try:
    from google.colab import drive
    drive.mount('/content/drive')
    OUTPUT = Path('/content/drive/MyDrive/AI-Crime-Investigation-FYP/canonical-four-class-v2')
except ImportError:
    OUTPUT = REPO / 'artifacts'
OUTPUT.mkdir(parents=True, exist_ok=True)
print('Persistent output:', OUTPUT)


In [ ]:
import kagglehub
try:
    KAGGLE_DOWNLOAD = Path(kagglehub.dataset_download('mateohervas/dcsass-dataset'))
except Exception as exc:
    raise RuntimeError('Kaggle download failed. Check network/access; if authentication is required, use Colab secrets or kagglehub.login(), never commit credentials.') from exc
from scripts.dataset_paths import resolve_dataset_root
DATA = resolve_dataset_root(KAGGLE_DOWNLOAD)
print('KaggleHub returned:', KAGGLE_DOWNLOAD)
print('Verified dataset package root:', DATA)


In [ ]:
from scripts.data import inspect
import json
report = inspect(DATA, OUTPUT / 'inspection')
print(json.dumps(report, indent=2))
assert report['raw_video_files'] == report['unique_video_clips'] + report['excluded_duplicate_copies']
assert sum(v['clips'] for v in report['mapped_counts'].values()) + report['unmapped_clips'] == report['unique_video_clips']
print('Review manifest.csv, excluded_duplicates.csv, and inspection.json in', OUTPUT / 'inspection')
print('Top-level entries:', [(p.name, 'directory' if p.is_dir() else 'file') for p in sorted(DATA.iterdir())])


In [ ]:
# Inspect label file contents before declaring bounding boxes or narratives absent.
for relative in report['annotation_candidates'][:20]:
    path = DATA / relative
    print(relative, path.read_text(errors='replace')[:2000])


Review all annotation candidates in Drive/inspection/inspection.json. Record actual schemas and whether temporal labels, object boxes, and paired scenario text exist. DCSASS category/temporal labels must not be treated as bounding boxes or narrative targets. If boxes are absent, obtain a separately licensed, human-annotated Kaggle/Roboflow detection dataset before detector training. No detector is trained here. Review manifest.csv source IDs and class mapping before notebook 02; unmapped categories are excluded. Never infer ground truth narrative scenarios from class names.
The user reported 33,278 raw rows / 16,639 unique clips / 520 original sources. These are user observations, not hard-coded expectations or results from this notebook. Compare this run's observed report. Fighting was reported to have only eight sources. Unknown source naming or related incidents with different IDs require manual review before splitting.
